# 12 -- Веса и архив

Это служебный ноутбук, не эксперимент. Если веса уже лежат в outputs/models, скачивание не нужно.
При подготовке с нуля беру revision из сохраненного manifest, а не последнюю версию модели.
Архив создаю отдельно от репозитория, без окружения, Git и восстановимых кешей.

In [1]:
from pathlib import Path
import json
import os
import zipfile

ar_root = Path.cwd() if (Path.cwd() / 'notebooks').exists() else Path.cwd().parent
get_ipython().run_line_magic('run', str(ar_root / 'notebooks/common.ipynb'))
ar_manifest = json.loads((ar_root / 'outputs/models/dense_manifest.json').read_text())
ar_model_path = ar_root / 'outputs/models/multilingual-e5-small'
print('Pinned revision:', ar_manifest['revision'])


Pinned revision: 614241f622f53c4eeff9890bdc4f31cfecc418b3


## Подготовка весов

Эту ячейку запускаю до 04, только если нет локальных весов. Она требует интернет для скачивания.
Дальнейшие ноутбуки работают локально. Пустой revision считаю ошибкой, чтобы случайно не взять другие веса.

In [2]:
assert ar_manifest.get('revision'), 'В manifest отсутствует revision модели'
if not (ar_model_path / 'model.safetensors').exists():
    from huggingface_hub import snapshot_download

    os.environ.pop('HF_HUB_OFFLINE', None)
    snapshot_download(
        ar_manifest['model'],
        revision=ar_manifest['revision'],
        local_dir=ar_model_path,
        allow_patterns=['*.json', '*.safetensors', '*.txt', 'README.md'],
        ignore_patterns=['onnx/*', 'openvino/*'],
        max_workers=3,
    )
print(ar_model_path)


/Users/dvsorochan/.codex/.chatgpt-projects/g-p-6ab7bad8eeb481919ab7a788d369d736/avito_candidate_generation/outputs/models/multilingual-e5-small


## Архив решения

Перед упаковкой выполняю проверки из 11. В архив включаю исходные данные и локальные веса,
поэтому размер около 1 GB. Исключаю только восстанавливаемые кеши и служебные файлы.
Исходные Parquet беру из AVITO_DATA_DIR или dataset/.

In [3]:
ar_output = ar_root.parent / 'avito_candidate_generation_final.zip'
ar_excluded = {
    '.git',
    '.venv',
    '.runtime',
    '__pycache__',
    '.cache',
    '.ipynb_checkpoints',
    'src',
    'scripts',
    'tests',
}
ar_files = []
for ar_path in sorted(ar_root.rglob('*')):
    if not ar_path.is_file():
        continue
    ar_relative = ar_path.relative_to(ar_root)
    if any(part in ar_excluded for part in ar_relative.parts):
        continue
    if (
        ar_relative.parts[:2] == ('outputs', 'cache')
        or ar_relative.parts[0] == 'dataset'
    ):
        continue
    if ar_path.suffix in ['.zip', '.pyc', '.log'] or ar_path.name == '.DS_Store':
        continue
    ar_files.append((ar_path, Path('avito_candidate_generation') / ar_relative))
for ar_name in [
    'train.parquet',
    'benchmark_queries.parquet',
    'benchmark_items.parquet',
]:
    ar_files.append(
        (data_dir() / ar_name, Path('avito_candidate_generation/dataset') / ar_name)
    )


In [4]:
with zipfile.ZipFile(
    ar_output, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=3, allowZip64=True
) as ar_zip:
    for ar_source, ar_target in ar_files:
        ar_zip.write(ar_source, str(ar_target))
with zipfile.ZipFile(ar_output) as ar_zip:
    assert ar_zip.testzip() is None
ar_result = {
    'path': str(ar_output),
    'files': len(ar_files),
    'bytes': ar_output.stat().st_size,
    'sha256': sha256_file(ar_output),
    'crc_valid': True,
}
write_json(ar_result, ar_output.with_suffix('.manifest.json'))
print(ar_result)


{'path': '/Users/dvsorochan/.codex/.chatgpt-projects/g-p-6ab7bad8eeb481919ab7a788d369d736/avito_candidate_generation_final.zip', 'files': 130, 'bytes': 984277272, 'sha256': 'ec2d4b004096c54c733bdbf07e7198fa3597523b10d179638919b7de0792b862', 'crc_valid': True}
